# Final test-set scores

This notebook scores the three crater detectors (U-Net run 2, Faster R-CNN, DINOv3 + head) on the **test set**.
Nothing is trained here. Each model uses the cutoff that was already chosen on the validation set.

## 1. Imports and setup

Load the libraries and pick the device (GPU if there is one).

In [1]:
import json
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from sklearn.model_selection import train_test_split
from torchvision.ops import box_iou
from tqdm import tqdm

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Using device:", device)

Using device: mps


## 2. Data split

The same split as every other notebook: 80% train, then the remaining 20% split in half into validation and test.
We only use `test_ids` here.

In [2]:
images_path = Path("dataset/images")
data = json.loads(Path("dataset/gt_public.json").read_text())

image_ids = list(data.keys())
train_ids, rest_ids = train_test_split(image_ids, test_size=0.2, random_state=0)
val_ids, test_ids = train_test_split(rest_ids, test_size=0.5, random_state=0)

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

Train: 41470, Val: 5184, Test: 5184


## 3. Shared scoring functions

These are the same functions used for the validation scores:

- `get_true_boxes`: turns the (x, y, w, h) labels into (x1, y1, x2, y2) boxes, like in 06-faster-rcnn.ipynb.
- `count_matches`: a predicted box finds a true crater if their box IoU is at least 0.5. Each true crater can be matched only once.
- `mask_to_boxes`: for U-Net and DINOv3, each connected group of crater pixels becomes one box.
- `get_scores`: precision, recall and F1, counted over all test images together.

In [3]:
def get_true_boxes(image_id):
    boxes = [[x, y, x + w, y + h] for x, y, w, h in data[image_id] if w > 0 and h > 0]
    return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)


def count_matches(pred_boxes, true_boxes, iou_threshold=0.5):
    if len(pred_boxes) == 0 or len(true_boxes) == 0:
        return 0

    ious = box_iou(pred_boxes, true_boxes)
    matches = 0
    for i in range(len(pred_boxes)):
        j = ious[i].argmax()
        if ious[i, j] >= iou_threshold:
            matches += 1
            ious[:, j] = -1  # this real crater is taken
    return matches


def mask_to_boxes(pred_mask):
    labeled, n = ndimage.label(pred_mask)
    boxes = []
    for ys, xs in ndimage.find_objects(labeled):
        boxes.append([xs.start, ys.start, xs.stop, ys.stop])
    return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)


def get_scores(tp, n_pred, n_true):
    precision = tp / n_pred if n_pred > 0 else 0.0
    recall = tp / n_true
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1


def save_scores(run_dir, cutoff, precision, recall, f1):
    scores = {"cutoff": cutoff, "precision": precision, "recall": recall, "f1": f1}
    (run_dir / "test_scores.json").write_text(json.dumps(scores, indent=2))
    print(f"precision {precision:.4f}, recall {recall:.4f}, F1 {f1:.4f}")

## 4. U-Net (run 2)

The model class is copied from 05-unet.ipynb. We load the saved weights, predict a crater mask for every test image,
keep pixels with probability > 0.9, turn the mask into boxes, and count matches.

In [4]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
    )


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # shrinking half
        self.down1 = conv_block(1, 32)      # (32, 256, 256)
        self.down2 = conv_block(32, 64)     # (64, 128, 128)
        self.down3 = conv_block(64, 128)    # (128, 64, 64)
        self.bottom = conv_block(128, 256)  # (256, 32, 32)

        # growing half
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)  # (128, 64, 64)
        self.conv3 = conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)   # (64, 128, 128)
        self.conv2 = conv_block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)    # (32, 256, 256)
        self.conv1 = conv_block(64, 32)

        self.out = nn.Conv2d(32, 1, kernel_size=1)  # (1, 256, 256)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        b = self.bottom(self.pool(d3))

        u3 = self.conv3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.conv2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.conv1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)

In [5]:
unet_dir = Path("src/unet/02-cosine-lr")
unet_cutoff = 0.9

unet = UNet().to(device)
unet.load_state_dict(torch.load(unet_dir / "unet.pt", map_location=device))
unet.eval()


# same image loading as the CraterDataset in 05-unet.ipynb (grayscale, 0 to 1)
def load_gray_image(image_id):
    image = Image.open(images_path / f"{image_id}.png").convert("L")
    image = torch.tensor(np.array(image), dtype=torch.float32) / 255.0
    return image.unsqueeze(0)


tp = 0
n_pred = 0
n_true = 0

for start in tqdm(range(0, len(test_ids), 16), mininterval=1.0):
    batch_ids = test_ids[start:start + 16]
    images = torch.stack([load_gray_image(i) for i in batch_ids]).to(device)

    with torch.no_grad():
        probs = torch.sigmoid(unet(images)).squeeze(1).cpu().numpy()

    for prob, image_id in zip(probs, batch_ids):
        true_boxes = get_true_boxes(image_id)
        pred_boxes = mask_to_boxes(prob > unet_cutoff)
        n_true += len(true_boxes)
        n_pred += len(pred_boxes)
        tp += count_matches(pred_boxes, true_boxes)

unet_precision, unet_recall, unet_f1 = get_scores(tp, n_pred, n_true)
save_scores(unet_dir, unet_cutoff, unet_precision, unet_recall, unet_f1)

  0%|          | 0/324 [00:00<?, ?it/s]

  2%|▏         | 5/324 [00:01<01:05,  4.87it/s]

  4%|▍         | 14/324 [00:02<00:43,  7.17it/s]

  7%|▋         | 24/324 [00:03<00:36,  8.23it/s]

 10%|█         | 33/324 [00:04<00:34,  8.44it/s]

 13%|█▎        | 43/324 [00:05<00:32,  8.71it/s]

... (25 outputs left out)

 97%|█████████▋| 313/324 [00:33<00:01,  9.29it/s]

100%|█████████▉| 323/324 [00:34<00:00,  9.24it/s]

100%|██████████| 324/324 [00:34<00:00,  9.39it/s]

precision 0.4813, recall 0.7028, F1 0.5713


## 5. Faster R-CNN

Built the same way as in 06-faster-rcnn.ipynb: ResNet-50 + FPN, `min_size=256`, and a new 2-class head.
Then we load the saved weights, keep predicted boxes with score > 0.8, and count matches.

In [6]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

frcnn_dir = Path("src/faster-rcnn/01-baseline")
frcnn_cutoff = 0.8

frcnn = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=256)
in_features = frcnn.roi_heads.box_predictor.cls_score.in_features
frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)

frcnn.load_state_dict(torch.load(frcnn_dir / "model.pt", map_location=device))
frcnn.to(device)
frcnn.eval()


# same image loading as the CraterDetectionDataset in 06-faster-rcnn.ipynb (RGB, 0 to 1)
def load_rgb_image(image_id):
    image = Image.open(images_path / f"{image_id}.png").convert("RGB")
    return torch.tensor(np.array(image), dtype=torch.float32).permute(2, 0, 1) / 255.0


tp = 0
n_pred = 0
n_true = 0

for start in tqdm(range(0, len(test_ids), 8), mininterval=1.0):
    batch_ids = test_ids[start:start + 8]
    images = [load_rgb_image(i).to(device) for i in batch_ids]

    with torch.no_grad():
        outputs = frcnn(images)

    for out, image_id in zip(outputs, batch_ids):
        true_boxes = get_true_boxes(image_id)
        pred_boxes = out["boxes"][out["scores"] > frcnn_cutoff].cpu()
        n_true += len(true_boxes)
        n_pred += len(pred_boxes)
        tp += count_matches(pred_boxes, true_boxes)

frcnn_precision, frcnn_recall, frcnn_f1 = get_scores(tp, n_pred, n_true)
save_scores(frcnn_dir, frcnn_cutoff, frcnn_precision, frcnn_recall, frcnn_f1)

  0%|          | 0/648 [00:00<?, ?it/s]

  0%|          | 1/648 [00:04<46:22,  4.30s/it]

  1%|          | 4/648 [00:05<12:39,  1.18s/it]

  1%|          | 7/648 [00:06<07:54,  1.35it/s]

  2%|▏         | 10/648 [00:07<06:07,  1.74it/s]

  2%|▏         | 13/648 [00:08<05:18,  2.00it/s]

... (167 outputs left out)

 99%|█████████▉| 643/648 [03:35<00:01,  3.32it/s]

100%|█████████▉| 647/648 [03:36<00:00,  3.33it/s]

100%|██████████| 648/648 [03:37<00:00,  2.98it/s]

precision 0.7892, recall 0.8047, F1 0.7969


## 6. DINOv3 + head

The backbone, image preprocessing, feature extraction and head are copied from 07-dinov3.ipynb.
We extract the test features once with the frozen backbone, run the saved head, keep pixels with probability > 0.9,
turn the mask into boxes, and count matches.

DINOv3 is a gated model on Hugging Face, so you must be logged in once on this computer (`hf auth login` in a terminal).

In [7]:
from transformers import AutoModel

model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
dino = AutoModel.from_pretrained(model_name).to(device)
dino.eval()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def load_image(image_id):
    image = Image.open(images_path / f"{image_id}.png").convert("RGB")
    image = torch.tensor(np.array(image), dtype=torch.float32).permute(2, 0, 1) / 255
    image = (image - mean) / std
    return image


skip = 1 + dino.config.num_register_tokens

def get_features(ids, batch_size=32):
    all_features = []
    for start in tqdm(range(0, len(ids), batch_size), mininterval=1.0):
        batch_ids = ids[start:start + batch_size]
        images = torch.stack([load_image(i) for i in batch_ids]).to(device)

        with torch.no_grad():
            output = dino(pixel_values=images)

        patches = output.last_hidden_state[:, skip:, :]
        patches = patches.reshape(-1, 16, 16, 384).permute(0, 3, 1, 2)
        all_features.append(patches.half().cpu())

    return torch.cat(all_features)


test_features = get_features(test_ids)
print(test_features.shape)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1110 22:46:25.007000 15043 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 25314.59it/s]

  0%|          | 0/162 [00:00<?, ?it/s]

  2%|▏         | 4/162 [00:01<00:42,  3.68it/s]

... (20 outputs left out)

 93%|█████████▎| 151/162 [00:24<00:01,  6.42it/s]

 98%|█████████▊| 158/162 [00:25<00:00,  6.41it/s]

100%|██████████| 162/162 [00:25<00:00,  6.29it/s]

torch.Size([5184, 384, 16, 16])


In [8]:
class DinoHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Conv2d(384, 1, kernel_size=1)

    def forward(self, x):
        x = self.out(x)
        x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)
        return x

In [9]:
dino_dir = Path("src/dinov3/01-baseline")
dino_cutoff = 0.9

head = DinoHead().to(device)
head.load_state_dict(torch.load(dino_dir / "model.pt", map_location=device))
head.eval()

tp = 0
n_pred = 0
n_true = 0

for start in tqdm(range(0, len(test_ids), 64), mininterval=1.0):
    batch_ids = test_ids[start:start + 64]

    with torch.no_grad():
        probs = torch.sigmoid(head(test_features[start:start + 64].float().to(device)))
    probs = probs.squeeze(1).cpu().numpy()

    for prob, image_id in zip(probs, batch_ids):
        true_boxes = get_true_boxes(image_id)
        pred_boxes = mask_to_boxes(prob > dino_cutoff)
        n_true += len(true_boxes)
        n_pred += len(pred_boxes)
        tp += count_matches(pred_boxes, true_boxes)

dino_precision, dino_recall, dino_f1 = get_scores(tp, n_pred, n_true)
save_scores(dino_dir, dino_cutoff, dino_precision, dino_recall, dino_f1)

  0%|          | 0/81 [00:00<?, ?it/s]

 46%|████▌     | 37/81 [00:01<00:01, 36.97it/s]

 99%|█████████▉| 80/81 [00:02<00:00, 40.23it/s]

100%|██████████| 81/81 [00:02<00:00, 39.81it/s]

precision 0.3061, recall 0.1581, F1 0.2085


## 7. Test scores vs validation scores

Validation F1 values are the ones reported earlier (src/comparison/crater_scores.csv).

In [10]:
results = [
    ["U-Net 2", unet_cutoff, unet_precision, unet_recall, unet_f1, 0.5818],
    ["Faster R-CNN", frcnn_cutoff, frcnn_precision, frcnn_recall, frcnn_f1, 0.7958],
    ["DINOv3", dino_cutoff, dino_precision, dino_recall, dino_f1, 0.2143],
]

print(f"{'Model':<14}{'Cutoff':>8}{'Test P':>9}{'Test R':>9}{'Test F1':>9}{'Val F1':>9}")
for name, cutoff, precision, recall, f1, val_f1 in results:
    print(f"{name:<14}{cutoff:>8}{precision:>9.4f}{recall:>9.4f}{f1:>9.4f}{val_f1:>9.4f}")

Model           Cutoff   Test P   Test R  Test F1   Val F1
U-Net 2            0.9   0.4813   0.7028   0.5713   0.5818
Faster R-CNN       0.8   0.7892   0.8047   0.7969   0.7958
DINOv3             0.9   0.3061   0.1581   0.2085   0.2143
